# Phase 1 — Data Loading & Cleaning
Builds `data/processed/ports_clean.csv` from WPI (primary source).

**Note:** WPI's `UN/LOCODE` column is present but blank for all rows, so we do NOT merge with the UN/LOCODE code-list. WPI alone has everything we need: lat/lon, depth, harbor size, facilities, country.

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

wpi = pd.read_csv(f"{RAW_DIR}/WPI/world_port_index.csv")
print("WPI shape:", wpi.shape)

WPI shape: (3824, 107)


In [2]:
# Select only the columns relevant to graph-building and routing
keep_cols = [
    'World Port Index Number', 'Main Port Name', 'Country Code',
    'World Water Body', 'Latitude', 'Longitude',
    'Harbor Size', 'Harbor Type', 'Harbor Use',
    'Channel Depth (m)', 'Anchorage Depth (m)', 'Cargo Pier Depth (m)',
    'Maximum Vessel Length (m)', 'Maximum Vessel Beam (m)', 'Maximum Vessel Draft (m)',
    'Facilities - Wharves', 'Facilities - Container', 'Facilities - Ro-Ro',
    'Facilities - Solid Bulk', 'Facilities - Liquid Bulk', 'Facilities - Breakbulk',
    'Cranes - Fixed', 'Cranes - Mobile', 'Cranes - Container',
    'Pilotage - Available', 'Tugs - Assistance'
]

ports = wpi[keep_cols].copy()

# Rename to clean snake_case for downstream code
ports = ports.rename(columns={
    'World Port Index Number': 'port_id',
    'Main Port Name': 'port_name',
    'Country Code': 'country',
    'World Water Body': 'water_body',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Harbor Size': 'harbor_size',
    'Harbor Type': 'harbor_type',
    'Harbor Use': 'harbor_use',
    'Channel Depth (m)': 'channel_depth_m',
    'Anchorage Depth (m)': 'anchorage_depth_m',
    'Cargo Pier Depth (m)': 'cargo_pier_depth_m',
    'Maximum Vessel Length (m)': 'max_vessel_length_m',
    'Maximum Vessel Beam (m)': 'max_vessel_beam_m',
    'Maximum Vessel Draft (m)': 'max_vessel_draft_m',
    'Facilities - Wharves': 'has_wharves',
    'Facilities - Container': 'has_container',
    'Facilities - Ro-Ro': 'has_roro',
    'Facilities - Solid Bulk': 'has_solid_bulk',
    'Facilities - Liquid Bulk': 'has_liquid_bulk',
    'Facilities - Breakbulk': 'has_breakbulk',
    'Cranes - Fixed': 'crane_fixed',
    'Cranes - Mobile': 'crane_mobile',
    'Cranes - Container': 'crane_container',
    'Pilotage - Available': 'pilotage_available',
    'Tugs - Assistance': 'tugs_assistance'
})

print(ports.shape)
ports.head(3)

(3824, 26)


,port_id,port_name,country,water_body,latitude,longitude,harbor_size,harbor_type,harbor_use,channel_depth_m,...,has_container,has_roro,has_solid_bulk,has_liquid_bulk,has_breakbulk,crane_fixed,crane_mobile,crane_container,pilotage_available,tugs_assistance
0,7950.0,Maurer,United States,North Atlantic Ocean,40.533333,-74.250000,Very Small,River (Natural),Unknown,11.0,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Yes
1,52235.0,Mangkasa Oil Terminal,Indonesia,Teluk Bone; Banda Sea; South Pacific Ocean,-2.733333,121.066667,Small,Open Roadstead,Unknown,9.4,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Yes,Yes
2,47620.0,Iharana,Madagascar,Indian Ocean,-13.350000,50.000000,Very Small,Coastal (Natural),Unknown,14.0,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Yes,Unknown


In [3]:
# --- Cleaning ---

# 1. Drop rows with missing/invalid coordinates (can't place on graph without these)
before = len(ports)
ports = ports.dropna(subset=['latitude', 'longitude'])
ports = ports[(ports['latitude'].between(-90, 90)) & (ports['longitude'].between(-180, 180))]
print(f"Dropped {before - len(ports)} rows with invalid/missing coordinates")

# 2. Drop exact duplicate ports (same name + country + coords)
before = len(ports)
ports = ports.drop_duplicates(subset=['port_name', 'country', 'latitude', 'longitude'])
print(f"Dropped {before - len(ports)} duplicate rows")

# 3. Standardize Yes/No/Unknown facility fields -> boolean-ish flag (1/0/-1 for unknown)
facility_cols = ['has_wharves', 'has_container', 'has_roro', 'has_solid_bulk',
                  'has_breakbulk', 'has_liquid_bulk', 'crane_fixed', 'crane_mobile',
                  'crane_container', 'pilotage_available', 'tugs_assistance']

def yn_to_flag(val):
    if pd.isna(val):
        return -1
    val = str(val).strip().lower()
    if val == 'yes':
        return 1
    elif val == 'no':
        return 0
    else:
        return -1  # Unknown

for col in facility_cols:
    ports[col] = ports[col].apply(yn_to_flag)

# 4. Depth/draft columns -> numeric, treat 0.0 as "not reported" -> NaN, then fill with column median
depth_cols = ['channel_depth_m', 'anchorage_depth_m', 'cargo_pier_depth_m',
              'max_vessel_length_m', 'max_vessel_beam_m', 'max_vessel_draft_m']

for col in depth_cols:
    ports[col] = pd.to_numeric(ports[col], errors='coerce')
    ports[col] = ports[col].replace(0.0, np.nan)

# Fill missing depth values with the median for that harbor_size group (small ports != big ports)
for col in depth_cols:
    ports[col] = ports.groupby('harbor_size')[col].transform(lambda x: x.fillna(x.median()))
    ports[col] = ports[col].fillna(ports[col].median())  # catch any remaining NaNs

# 5. Harbor size/type/use -> fill missing with 'Unknown'
for col in ['harbor_size', 'harbor_type', 'harbor_use', 'water_body', 'country', 'port_name']:
    ports[col] = ports[col].fillna('Unknown')

print(ports.isnull().sum())

Dropped 0 rows with invalid/missing coordinates
Dropped 0 duplicate rows
port_id                0
port_name              0
country                0
water_body             0
latitude               0
longitude              0
harbor_size            0
harbor_type            0
harbor_use             0
channel_depth_m        0
anchorage_depth_m      0
cargo_pier_depth_m     0
max_vessel_length_m    0
max_vessel_beam_m      0
max_vessel_draft_m     0
has_wharves            0
has_container          0
has_roro               0
has_solid_bulk         0
has_liquid_bulk        0
has_breakbulk          0
crane_fixed            0
crane_mobile           0
crane_container        0
pilotage_available     0
tugs_assistance        0
dtype: int64


In [4]:
# Final check + save
print("Final shape:", ports.shape)
print(ports.dtypes)
ports.head(5)

Final shape: (3824, 26)
port_id                float64
port_name                  str
country                    str
water_body                 str
latitude               float64
longitude              float64
harbor_size                str
harbor_type                str
harbor_use                 str
channel_depth_m        float64
anchorage_depth_m      float64
cargo_pier_depth_m     float64
max_vessel_length_m    float64
max_vessel_beam_m      float64
max_vessel_draft_m     float64
has_wharves              int64
has_container            int64
has_roro                 int64
has_solid_bulk           int64
has_liquid_bulk          int64
has_breakbulk            int64
crane_fixed              int64
crane_mobile             int64
crane_container          int64
pilotage_available       int64
tugs_assistance          int64
dtype: object


,port_id,port_name,country,water_body,latitude,longitude,harbor_size,harbor_type,harbor_use,channel_depth_m,...,has_container,has_roro,has_solid_bulk,has_liquid_bulk,has_breakbulk,crane_fixed,crane_mobile,crane_container,pilotage_available,tugs_assistance
0,7950.0,Maurer,United States,North Atlantic Ocean,40.533333,-74.250000,Very Small,River (Natural),Unknown,11.0,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,1
1,52235.0,Mangkasa Oil Terminal,Indonesia,Teluk Bone; Banda Sea; South Pacific Ocean,-2.733333,121.066667,Small,Open Roadstead,Unknown,9.4,...,-1,-1,-1,-1,-1,-1,-1,-1,1,1
2,47620.0,Iharana,Madagascar,Indian Ocean,-13.350000,50.000000,Very Small,Coastal (Natural),Unknown,14.0,...,-1,-1,-1,-1,-1,-1,-1,-1,1,-1
3,47360.0,Andoany,Madagascar,Mozambique Channel; Indian Ocean,-13.400000,48.300000,Very Small,Open Roadstead,Unknown,20.1,...,-1,-1,-1,-1,-1,1,1,-1,0,1
4,47020.0,Chake Chake,Tanzania,Indian Ocean,-5.250000,39.766667,Small,Coastal (Natural),Unknown,14.0,...,-1,-1,-1,-1,-1,-1,-1,-1,1,0


In [5]:
output_path = f"{PROCESSED_DIR}/ports_clean.csv"
ports.to_csv(output_path, index=False)
print(f"Saved {len(ports)} ports to {output_path}")

Saved 3824 ports to ../data/processed/ports_clean.csv
